# Ali Express Customer Life Time Value

In [ ]:
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns
from lifetimes import BetaGeoFitter, GammaGammaFitter, plotting
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from lifetimes.plotting import plot_frequency_recency_matrix, plot_probability_alive_matrix, plot_period_transactions
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.pipeline import Pipeline
import pandas as pd

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)


In [ ]:
df = pd.read_csv('AliExpress.csv', encoding='ISO-8859-1')
df.dropna(inplace=True)
df

# Data Visulization and EDA

In [ ]:
# Bar chart for Order Priority
plt.figure(figsize=(10, 6))
sns.countplot(x='Order Priority', data=df)
plt.title('Order Priority Distribution')
plt.show()

# Scatter plot for Discount vs. Sales
plt.figure(figsize=(8, 8))
sns.scatterplot(x='Discount', y='Sales', data=df)
plt.title('Discount vs. Sales')
plt.show()

# Pie chart for Product Category distribution
plt.figure(figsize=(8, 8))
df['Product Category'].value_counts().plot.pie(autopct='%1.1f%%')
plt.title('Product Category Distribution')
plt.show()

# Stacked bar chart for Product Category and Product Sub-Category
plt.figure(figsize=(12, 6))
df.groupby(['Product Category', 'Product Sub-Category']).size().unstack().plot(kind='bar', stacked=True)
plt.title('Product Category and Sub-Category Distribution')
plt.show()

# Region Distribution
plt.figure(figsize=(10, 6))
sns.countplot(x='Region', data=df)
plt.title('Region Distribution')
plt.xlabel('Region')
plt.ylabel('Count')
plt.show()


plt.figure(figsize=(12, 6))
sns.countplot(x='State or Province', data=df)
plt.title('State or Province Distribution')
plt.xlabel('State or Province')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.show()


plt.figure(figsize=(8, 6))
sns.countplot(x='Ship Mode', data=df)
plt.title('Ship Mode Distribution')
plt.xlabel('Ship Mode')
plt.ylabel('Count')
plt.show()

customer_segment_counts = df['Customer Segment'].value_counts()

plt.figure(figsize=(8, 8))
plt.pie(customer_segment_counts, labels=customer_segment_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Customer Segment Distribution')
plt.show()

# Line chart for Sales ove# Data Visulization and EDAr time
df['Order Date'] = pd.to_datetime(df['Order Date'])
plt.figure(figsize=(12, 6))
sns.lineplot(x='Order Date', y='Sales', data=df)
plt.title('Sales Over Time')
plt.show()



In [ ]:
df.describe()

# Data Processing Feature Extraction and Clipping Outliers

In [ ]:
df = df[df['Unit Price'] > 0]
df = df[df['Quantity ordered new'] > 0]
df

In [ ]:
df.dropna(inplace=True)

In [ ]:
def find_boundaries(df, variable,q1=0.05,q2=0.95):
    # the boundaries are the quantiles
    lower_boundary = df[variable].quantile(q1)
    upper_boundary = df[variable].quantile(q2)
    return upper_boundary, lower_boundary

def capping_outliaers(df,variable):
    upper_boundary,lower_boundary =  find_boundaries(df,variable)
    df[variable] = np.where(df[variable] > upper_boundary, upper_boundary,
                       np.where(df[variable] < lower_boundary, lower_boundary, df[variable]))

In [ ]:
capping_outliaers(df,'Unit Price')
capping_outliaers(df,'Product Base Margin')
capping_outliaers(df,'Shipping Cost')
capping_outliaers(df,'Quantity ordered new')
capping_outliaers(df,'Profit')

df.describe()

In [ ]:
df.describe()

In [ ]:
df['Order Date'] = pd.to_datetime(df['Order Date'], infer_datetime_format=True)
df

In [ ]:
df['Total Price']=df['Unit Price']*df['Quantity ordered new']

In [ ]:
df

# RMF Scoring

In [ ]:
import lifetimes

clv = lifetimes.utils.summary_data_from_transaction_data(df, 'Customer ID', 'Order Date', 'Total Price')


In [ ]:
clv

In [ ]:
clv = clv[clv['frequency']>1] # we want only customers shopped more than 2 times

In [ ]:
clv

# Expected Transactions Determination

In [ ]:
bgf = BetaGeoFitter(penalizer_coef=0.001)
bgf.fit(clv['frequency'], clv['recency'], clv['T'])

In [ ]:
# Assuming 'T' is the customer's age at the time of the last observation
t = clv['T'].max()
clv['expected_purchases_all_time'] = bgf.conditional_expected_number_of_purchases_up_to_time(t, clv['frequency'], clv['recency'], clv['T'])
top_customers = clv.sort_values(by='expected_purchases_all_time', ascending=False) 


In [ ]:
clv

# Expected Value Determination

In [ ]:
ggf = GammaGammaFitter(penalizer_coef=0.01)
ggf.fit(clv["frequency"],
        clv["monetary_value"])

In [ ]:
clv['All-time']=ggf.customer_lifetime_value(bgf,
                                   clv["frequency"],
                                   clv["recency"],
                                   clv["T"],
                                   clv["monetary_value"],
                                   time=t,
                                   freq='D',
                                   discount_rate=0.01)

In [ ]:
 clv.sort_values('All-time',ascending=False).head()

# Customer Segmentation

In [ ]:
clv['Segment'] =  pd.qcut(clv['All-time'],4,labels = ['Inactive','Requires Attension',
                                                          'Loyal Customers','Gold Members (Premiun)'])

In [ ]:
clv

# Regressor Training/Evalaution for Lifetime Value Prediction

In [ ]:
# Separate features and target variable for CLV prediction
clv_features = clv[['frequency', 'recency', 'T', 'monetary_value' ]]
clv_target = clv['All-time']

# Split the data into training and testing sets for CLV prediction
clv_X_train, clv_X_test, clv_y_train, clv_y_test = train_test_split(clv_features, clv_target, test_size=0.2, random_state=42)

# Train a Linear Regression model for CLV prediction
clv_model = LinearRegression()
clv_model.fit(clv_X_train, clv_y_train)

# Make predictions on the test set
clv_predictions = clv_model.predict(clv_X_test)

# Evaluate the CLV prediction model
mse = mean_squared_error(clv_y_test, clv_predictions)
print(f'Mean Squared Error for CLV Prediction: {mse}')

In [ ]:
 
# Standardize features for Random Forest Regressor
scaler = StandardScaler()# Classifier Training/Evalaution for Customer Segmentation
clv_X_train_scaled = scaler.fit_transform(clv_X_train)
clv_X_test_scaled = scaler.transform(clv_X_test)

# Train a Random Forest Regressor for CLV prediction
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(clv_X_train_scaled, clv_y_train)

# Make predictions on the test set
clv_predictions_rf = rf_model.predict(clv_X_test_scaled)

# Evaluate the Random Forest Regressor model
mse_rf = mean_squared_error(clv_y_test, clv_predictions_rf)
print(f'Mean Squared Error for CLV Prediction (Random Forest): {mse_rf}')

# Classifier Training/Evalaution for Customer Segmentation

In [ ]:
# Separate features and target variable for Segment prediction
segment_features = clv[['frequency', 'recency', 'T', 'monetary_value', 'All-time']]
segment_target = clv['Segment']

# Split the data into training and testing sets for Segment prediction
segment_X_train, segment_X_test, segment_y_train, segment_y_test = train_test_split(segment_features, segment_target, test_size=0.2, random_state=42)

# Create a pipeline for preprocessing and training the Random Forest Classifier
segment_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Train the Random Forest Classifier for Segment prediction
segment_pipeline.fit(segment_X_train, segment_y_train)

# Make predictions on the test set
segment_predictions = segment_pipeline.predict(segment_X_test)

# Evaluate the Segment prediction model
accuracy = accuracy_score(segment_y_test, segment_predictions)
print(f'Accuracy for Segment Prediction: {accuracy}')
# Classifier Training/Evalaution for Customer Segmentation
# Display classification report for more detailed metrics
print(classification_report(segment_y_test, segment_predictions))

# Saving Models

In [ ]:
import joblib

# Save Random Forest Regressor model for CLV prediction
joblib.dump(rf_model, 'clv_rf_model.joblib')

# Save Segment prediction pipeline
joblib.dump(segment_pipeline, 'segment_pipeline.joblib')

scaler_filename = 'standard_scaler.joblib'
joblib.dump(scaler, scaler_filename)
print(f'StandardScaler object saved to {scaler_filename}')

# Prediction Fucntions

In [ ]:
def predict_clv(new_customer_features):
    # Ensure that the new customer features are in the correct order
    new_customer_features = pd.DataFrame([new_customer_features], columns=['frequency', 'recency', 'T', 'monetary_value'])
# Saving Models
    # Standardize the features for the Random Forest Regressor
    new_customer_features_scaled = scaler.transform(new_customer_features)

    # Predict CLV using the Random Forest Regressor
    clv_prediction = rf_model.predict(new_customer_features_scaled)[0]
    return clv_prediction

# Example usage:
new_customer_features = [5888, 1000, 11500, 500000]  # Replace with the actual features of the new customer
predicted_clv = predict_clv(new_customer_features)

print(f'Predicted Segment: {predicted_clv}')


In [ ]:
def predict_segment(new_customer_features):
    # Ensure that the new customer features are in the correct order
    new_customer_features = pd.DataFrame([new_customer_features], columns=['frequency', 'recency', 'T', 'monetary_value', 'All-time'])

    # Use the trained pipeline to standardize features and predict segment
    new_customer_segment = segment_pipeline.predict(new_customer_features)[0]
    return new_customer_segment

# Example usage:
new_customer_features = [5888, 1000, 11500, 500,7000]  # Replace with the actual features of the new customer
predicted_segment = predict_segment(new_customer_features)

print(f'Predicted Segment: {predicted_segment}')


End of Notebook